<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 22 — Pipeline ו־Cross Validation
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
בהשוואת שיטות לא נרצה שמועמד אחד ייבחן בשאלות קלות ואחר בקשות. נשווה באותם קיפולים ונלמד הכנה מחדש בכל קיפול.

## מטרות
- להסביר קיפולי CV.
- למקם הכנה ב־Pipeline.
- לבחור ולהתאים מחדש.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## הנחיות למורה
זהו פתרון מתועד למעבדה. מבדילים בין תוצאה נכונה לבין נימוק נכון, ובודקים שהקוד תואם לנתוני התלמיד. המקורות הישנים נשמרו ללא שינוי.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## פתרון חלק 1
הפתרון הבא תואם לנתוני התרגול העצמאי, ולא לנתוני הדוגמה הפתורה.

</div>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

<div dir="rtl" lang="he" style="text-align: right;">

## 1. חלוקה
Train פנימי לומד הכנה ומודל; Validation בוחר K; Test חיצוני מדווח ביצוע לאחר הבחירה. K הוא מספר שכנים, היפר־פרמטר. אין להשלים או לתקנן מראש את כל קבוצת האימון החיצונית, כי בכל סבב חלקה משמש לאימות.

</div>

In [ ]:
url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
model_df = df.dropna(subset=["species"]).copy()
features = ["bill_length_mm","bill_depth_mm"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print("Outer Train:", len(X_train), "Outer Test:", len(X_test))
display(X_train.isna().sum())

<div dir="rtl" lang="he" style="text-align: right;">

## 2. תהליך CV
ה־ Pipeline מתאים מחדש השלמה, תקנון ומודל באימון של כל קיפול. cross_validate מקבל X_train, y_train בלבד. השדה test_score שלו מתייחס לאימות הפנימי; הוא אינו מכיל את ציון Test החיצוני.

</div>

In [ ]:
# ההגדרות נקבעות לפני צפייה בהערכת Test
candidates = [1, 3, 5, 15, 51]
metric = "f1_macro"
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

# קוד תהליך נתון: המטרה להבין מה נכנס ל-CV ומה נשאר בחוץ
for k in candidates:
    pipe = make_pipeline(
        SimpleImputer(strategy="mean"),
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k, weights="uniform", metric="euclidean")
    )
    fold_results = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring=metric, return_train_score=True, n_jobs=1
    )
    # test_score כאן הוא Validation פנימי של הקיפול, לא ה-Test החיצוני
    results.append([
        k,
        fold_results["train_score"].mean(),
        fold_results["test_score"].mean(),
        fold_results["test_score"].std(ddof=0)
    ])

results_df = pd.DataFrame(results, columns=["K", "train_mean", "cv_mean", "cv_std"])
display(results_df)

<div dir="rtl" lang="he" style="text-align: right;">

## 3. בחירה
השורה עם cv_mean הגבוה ביותר נבחרת לפי הכלל שהוגדר מראש. פער בין אימון לאימות יכול להצביע על חשש להתאמת יתר, אבל אינו הוכחה יחידה; אי אפשר לדרוש שהציונים ישתנו באופן מונוטוני עם K.

</div>

In [ ]:
# כלל שנקבע מראש: ממוצע CV הגבוה ביותר; בשוויון מדויק K הקטן יותר
# המיון והגישה לשורה הם קטע עזר נתון
ranked = results_df.sort_values(["cv_mean", "K"], ascending=[False, True])
best_k = int(ranked.iloc[0]["K"])
print("Selected K by CV only:", best_k)

# מתאימים צינור חדש לכל קבוצת האימון החיצונית
final_model = make_pipeline(
    SimpleImputer(strategy="mean"),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=best_k, weights="uniform", metric="euclidean")
)
final_model.fit(X_train, y_train)

<div dir="rtl" lang="he" style="text-align: right;">

## 4. הערכה
המודל הסופי הותאם מחדש לכל Train, ולא נבחר מבין חמשת מודלי הקיפולים. כעת מפיקים תחזיות פעם אחת, ומדווחים על אותה סדרת תחזיות בכמה מדדים.

</div>

In [ ]:
# רק לאחר בחירת K והתאמה מחדש: הערכה אחת על Test החיצוני
predictions = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, predictions)
labels_species = ["Adelie", "Chinstrap", "Gentoo"]
report = classification_report(
    y_test, predictions, labels=labels_species,
    output_dict=True, zero_division=0
)
report_table = pd.DataFrame(report).T
display(report_table.loc[labels_species, ["precision", "recall", "f1-score", "support"]])
test_macro_f1 = report["macro avg"]["f1-score"]
print("CV selection metric:", metric)
print("Selected model Test macro-F1:", test_macro_f1)
print("Selected model Test Accuracy:", test_accuracy)

<div dir="rtl" lang="he" style="text-align: right;">

## 5. כרטיס ניסוי ומגבלות
מועמדים [1,3,5,15,51], macro-F1, חלוקה 75%/25%, seed=17 ו־ stratify, חמישה קיפולים עם seed=42. ההכרעה היא ממוצע CV גבוה ביותר ושוויון לפי K קטן. המדד ותהליך ההכרעה נבחרו לפני Test; טבלת התוצאות קובעת את K בפועל.
הכנה על כל X_train לפני CV תשתמש במידע מתוך רשומות האימות הפנימי לחישוב ממוצעים ותקנון. Pipeline בתוך CV מונע זאת. cv_std מתאר פיזור חמשת הציונים ואינו רווח סמך; הקיפולים גם אינם חמישה ניסויים עצמאיים לחלוטין.
Penguins כבר נראה בחומרים הקודמים, לכן זו הדגמת תהליך ולא אומדן עצמאי חדש של הכללה. שינוי seed אינו מחזיר מצב של נתונים שמעולם לא נראו.
כאשר יש כמה רשומות לאותו אדם, משאירים את כל הרשומות שלו באותו צד של החלוקה, וגם בקיפולים הפנימיים משתמשים בחלוקה לפי קבוצות. אחרת המודל עשוי לנצל זיהוי עקיף של האדם במקום להכליל לאחרים.
אין להשוות את ציון מחברת התרגול למודרכת כהוכחת עדיפות מאפיינים: גם החלוקה שונה. בפרויקט עתידי נועלים מראש את מתכון ההערכה ומתעדים חריגות.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## תשובה לכרטיס היציאה
גם נתוני האימות הפנימיים משפיעים על ההכנה. לכן fit של ההכנה מתבצע בכל תת־אימון מחדש.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## משוב דידקטי
בדקו בנפרד: בחירת פעולה, תקינות קוד, פירוש פלט, יחידות ומגבלה. שאלו מה ישתנה בקלט חדש. אם יש קושי, חזרו לדוגמה קטנה לפני הגדלת המאגר. זהו משוב מקומי, לא מחוון ניקוד רשמי.

</div>